In [ ]:
# ===============================================
# 🔹 recipe_by_type.csv 데이터 전처리 (요구사항 완벽 반영)
# ===============================================

import pandas as pd
import numpy as np
import re

# ---------------------------
# 1. CSV 불러오기
# ---------------------------
df = pd.read_csv("recipe_by_type.csv")

# ---------------------------
# 2. 텍스트 정리 함수 정의
# ---------------------------
def clean_text(text):
    if pd.isna(text):
        return text
    text = str(text)
    # 긴 공백 제거
    text = re.sub(r"\s+", " ", text)
    # 불필요한 특수문자 제거 (~,~~,^^,\n,..,...,♥,+,_, 등)
    text = re.sub(r"[~\^♥\+\_]", "", text)
    text = text.replace("~~", "").replace("^^", "").replace("\n", "").replace("...", "").replace("..", "")
    return text.strip()

# ---------------------------
# 3. 긴 공백 / 특수문자 전처리 (ingredient_full, step_text, tag)
# ---------------------------
for col in ["ingredient_full", "step_text", "tag"]:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

# ---------------------------
# 4. servings 채우기 (cooking_time + step_text 기반)
# ---------------------------
def infer_servings(row):
    if not pd.isna(row["servings"]):
        return row["servings"]

    step = str(row.get("step_text", "")).lower()
    cook_time = row.get("cooking_time", np.nan)

    # step_text 기반 패턴
    if any(x in step for x in ["1인분", "혼자", "한 명", "한사람"]):
        return 1
    elif any(x in step for x in ["2인분", "둘이", "2명", "두 명"]):
        return 2
    elif any(x in step for x in ["3인분", "셋이", "3명"]):
        return 3
    elif any(x in step for x in ["4인분", "4명", "가족", "여럿"]):
        return 4
    # cooking_time 기준 추론
    elif pd.notna(cook_time):
        if cook_time <= 15:
            return 1
        elif cook_time <= 30:
            return 2
        elif cook_time <= 60:
            return 3
        else:
            return 4
    return np.nan

df["servings"] = df.apply(infer_servings, axis=1)

# ---------------------------
# 5. cooking_time 채우기 (servings + step_text 기반)
# ---------------------------
def infer_cooking_time(row):
    if not pd.isna(row["cooking_time"]):
        return row["cooking_time"]

    step = str(row.get("step_text", "")).lower()
    serv = row.get("servings", np.nan)

    # step_text 내 시간 단서 추출
    match = re.search(r"(\d+)\s*(분|시간)", step)
    if match:
        val = int(match.group(1))
        if "시간" in match.group(2):
            return val * 60
        else:
            return val

    # servings 기반 추론
    if not pd.isna(serv):
        if serv <= 1:
            return 15
        elif serv <= 2:
            return 30
        elif serv <= 4:
            return 45
        else:
            return 60
    return np.nan

df["cooking_time"] = df.apply(infer_cooking_time, axis=1)

# ---------------------------
# 6. level_nm 변환 (규칙 완전 반영)
# ---------------------------
def map_level(row):
    level = str(row.get("level_nm", "")).strip()
    cook_time = row.get("cooking_time", np.nan)
    serv = row.get("servings", np.nan)

    # 기존 매핑
    if level == "초급":
        return "하"
    elif level == "중급":
        return "상"
    elif level == "아무나":
        if pd.notna(cook_time):
            return "상" if cook_time >= 30 else "하"
        else:
            return "하"

    # 결측치 처리
    if level in ["", "nan", "None"] or pd.isna(level):
        if pd.notna(serv) and serv >= 5:
            return "상"
        else:
            return "하"

    return level

df["level_nm"] = df.apply(map_level, axis=1)

# ---------------------------
# 7. 인덱스 재정렬 (중간 빈행 제거)
# ---------------------------
df.dropna(how="all", inplace=True)
df.reset_index(drop=True, inplace=True)

# ---------------------------
# 8. 저장
# ---------------------------
output_path = "recipe_by_type_cleaned.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✅ 전처리 완료! 저장된 파일: {output_path}")

✅ 전처리 완료! 'recipe_by_type_cleaned.csv' 파일이 생성되었습니다.
level_nm NaN 개수: 0


## 인덱스 재정렬

In [2]:
import pandas as pd

# CSV 파일 경로
path = r"C:\githome\Personalized_healthcare_Project\recipe-frontend\recipe.csv"

# CSV 불러오기
df = pd.read_csv(path)

# ✅ 완전히 빈 행 제거
df.dropna(how="all", inplace=True)

# ✅ 인덱스 초기화
df.reset_index(drop=True, inplace=True)

# ✅ recipe_id 컬럼을 1부터 순차적으로 다시 부여
df['recipe_id'] = range(1, len(df) + 1)

# ✅ 저장 (덮어쓰기)
df.to_csv(path, index=False, encoding='utf-8-sig')

print(f"✅ recipe_id를 1부터 {len(df)}까지 재정렬 완료!")


✅ recipe_id를 1부터 1794까지 재정렬 완료!


In [ ]:
# ===============================================
# 🔹 recipe.csv 전처리 코드 (v2)
# ===============================================

import pandas as pd
import re

# ---------------------------
# 1️⃣ CSV 파일 로드
# ---------------------------
file_path = r"C:\githome\Personalized_healthcare_Project\recipe-frontend\recipe.csv"
df = pd.read_csv(file_path)

print("✅ 원본 데이터 크기:", df.shape)

# ---------------------------
# 2️⃣ 불필요한 레시피 행 삭제
# ---------------------------
delete_keywords = ["법", "손질", "보관", "삶기"]

pattern = "|".join(delete_keywords)
df = df[~df["recipe_nm_ko"].astype(str).str.contains(pattern, regex=True, na=False)]

print("🧹 불필요한 행 삭제 후 크기:", df.shape)

# ---------------------------
# 3️⃣ 텍스트 정제 함수 정의
# ---------------------------
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    remove_chars = ["null", "@", "+", "➊", "➋", "➌", "➍", "➎", "➏", "\n", "<", ">", "-", "#"]
    pattern = "|".join(map(re.escape, remove_chars))
    cleaned = re.sub(pattern, "", text)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

# ---------------------------
# 4️⃣ 세 컬럼에 대해 텍스트 정제 적용
# ---------------------------
target_cols = ["ingredient_full", "step_text", "tag"]

for col in target_cols:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)
        print(f"✨ {col} 컬럼 정제 완료")

# ---------------------------
# 5️⃣ 인덱스 재정렬 후 저장
# ---------------------------
df.reset_index(drop=True, inplace=True)

output_path = r"C:\githome\Personalized_healthcare_Project\recipe-frontend\recipe_cleaned_v2.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✅ 전처리 완료! 저장 파일명: {output_path}")
print("📊 최종 데이터 크기:", df.shape)


✅ 원본 데이터 크기: (1794, 9)
🧹 불필요한 행 삭제 후 크기: (1701, 9)
✨ ingredient_full 컬럼 정제 완료
✨ step_text 컬럼 정제 완료
✨ tag 컬럼 정제 완료
✅ 전처리 완료! 저장 파일명: C:\githome\Personalized_healthcare_Project\recipe-frontend\recipe_cleaned_v2.csv
📊 최종 데이터 크기: (1701, 9)
